# Setup Ollama

Lanch xtrem terminal in window.

%xterm

Download and run ollama server

curl -sSL https://ollama.ai/install.sh | sh && ollama serve

In [ ]:
!pip install colab-xterm
%load_ext colabxterm
%xterm

In [ ]:
!curl http://localhost:11434/api/pull -d '{  "model": "nomic-embed-text" }'

In [ ]:
!curl http://localhost:11434/api/pull -d '{  "model": "qwen2.5:7b" }'

# Setup

In [ ]:
#!pip install -qU langchain-community langchain-ollama langchain-openai pypdf faiss-cpu duckduckgo-search

Instalowane pakiety to:
- `langchain-community` - biblioteka zawierająca komponenty społecznościowe dla ekosystemu LangChain
- `langchain-openai` - integracja LangChain z API OpenAI
- `langchain-ollama` - integracja LangChain z API Ollama
- `pypdf` - biblioteka do pracy z plikami PDF
- `faiss-cpu` - wydajna biblioteka do przeszukiwania wektorów (implementacja CPU)
- `duckduckgo-search` - interfejs do korzystania z wyszukiwarki DuckDuckGo


In [1]:
# Core dependencies
import json
from typing import List, Tuple

# LangChain dependencies
from langchain_core.embeddings import Embeddings
from langchain_core.language_models.chat_models import BaseChatModel

from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_ollama import ChatOllama, OllamaEmbeddings
from pydantic import BaseModel, Field
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.tools import DuckDuckGoSearchResults

# Environment setup
# from google.colab import userdata


# os.environ["OPENAI_API_KEY"] = userdata.get('openaivision')

- `PromptTemplate` - pozwala na tworzenie szablonów instrukcji dla modeli językowych
- `ChatOpenAI` i `OpenAIEmbeddings` - umożliwiają komunikację z modelami OpenAI oraz tworzenie wektorowych reprezentacji tekstu
- `BaseModel` i `Field` z Pydantic - służą do tworzenia struktur danych z walidacją typów
- `PyPDFLoader` - narzędzie do wczytywania dokumentów PDF
- `RecursiveCharacterTextSplitter` - dzieli długie teksty na mniejsze fragmenty
- `FAISS` - tworzy przeszukiwalną bazę wektorów tekstowych
- `DuckDuckGoSearchResults` - narzędzie do wyszukiwania informacji w internecie

In [2]:
class CFG:
    path = "./content/Understanding_Climate_Change.pdf"
    model_openai = "gpt-4o-mini"
    model_ollama = "qwen2.5:7b"
    model_embeddings_ollama = "nomic-embed-text"
    max_tokens = 1000
    temperature = 0

# Funkcje

In [3]:
search = DuckDuckGoSearchResults()

In [4]:
def encode_pdf(path, embeddings: Embeddings, chunk_size=1000, chunk_overlap=200):
    # Load PDF documents
    loader = PyPDFLoader(path)
    documents = loader.load()

    # Split documents into chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len
    )
    cleaned_texts = text_splitter.split_documents(documents)

    # Create embeddings and vector store
    vectorstore = FAISS.from_documents(cleaned_texts, embeddings)

    return vectorstore


In [5]:
# Retrieval Evaluator
class RetrievalEvaluatorInput(BaseModel):
    relevance_score: float = Field(
        ...,
        description="The relevance score of the document to the query. the score should be between 0 and 1.",
    )


def retrieval_evaluator(query: str, document: str, llm: BaseChatModel) -> float:
    prompt = PromptTemplate(
        input_variables=["query", "document"],
        template="On a scale from 0 to 1, how relevant is the following document to the query? Query: {query}\nDocument: {document}\nRelevance score:",
    )
    chain = prompt | llm.with_structured_output(RetrievalEvaluatorInput)
    input_variables = {"query": query, "document": document}
    result = chain.invoke(input_variables).relevance_score
    return result

Ten fragment kodu wprowadza mechanizm oceny trafności wyników wyszukiwania informacji. Składa się z dwóch głównych części: definicji modelu danych i funkcji ewaluacyjnej.

W pierwszej części zdefiniowano klasę `RetrievalEvaluatorInput`, która dziedziczy po `BaseModel` z biblioteki Pydantic. Klasa ta ma tylko jedno pole:
- `relevance_score` - liczba zmiennoprzecinkowa określająca, jak bardzo dokument jest istotny dla zapytania.

Druga część to definicja funkcji `retrieval_evaluator`, która przyjmuje dwa parametry tekstowe:
- `query` - zapytanie użytkownika
- `document` - dokument, którego trafność względem zapytania chcemy ocenić

In [6]:
# Knowledge Refinement
class KnowledgeRefinementInput(BaseModel):
    key_points: str = Field(
        ..., description="The document to extract key information from."
    )


def knowledge_refinement(document: str, llm: BaseChatModel) -> List[str]:
    prompt = PromptTemplate(
        input_variables=["document"],
        template="Extract the key information from the following document in bullet points:\n{document}\nKey points:",
    )
    chain = prompt | llm.with_structured_output(KnowledgeRefinementInput)
    input_variables = {"document": document}
    result = chain.invoke(input_variables).key_points
    return [point.strip() for point in result.split("\n") if point.strip()]


Ten fragment kodu implementuje mechanizm ekstrakcji kluczowych informacji z dokumentu. Jest to istotny element w procesie przetwarzania większych tekstów, pomagający wyodrębnić najważniejsze punkty.

1. Klasa `KnowledgeRefinementInput` dziedziczy po `BaseModel` z biblioteki Pydantic i definiuje strukturę danych dla wyników ekstrakcji. Zawiera ona jedno pole:
   - `key_points`: łańcuch tekstowy przechowujący wyekstrahowane kluczowe informacje z dokumentu

In [7]:
# Web Search Query Rewriter
class QueryRewriterInput(BaseModel):
    query: str = Field(..., description="The query to rewrite.")


def rewrite_query(query: str, llm: BaseChatModel) -> str:
    prompt = PromptTemplate(
        input_variables=["query"],
        template="Rewrite the following query to make it more suitable for a web search:\n{query}\nRewritten query:",
    )
    chain = prompt | llm.with_structured_output(QueryRewriterInput)
    input_variables = {"query": query}
    return chain.invoke(input_variables).query.strip()

Ten fragment kodu wprowadza mechanizm przepisywania zapytań do wyszukiwania w internecie. Jest to ważny element poprawiający skuteczność wyszukiwania informacji, ponieważ zapytania użytkowników często nie są sformułowane w sposób optymalny dla wyszukiwarek.

1. Klasa `QueryRewriterInput` jest modelem danych wykorzystującym Pydantic. Definiuje ona strukturę, w jakiej przechowywane będzie przepisane zapytanie. Zawiera jedno pole:
   - `query` - łańcuch tekstowy przechowujący przepisane zapytanie
   
   Pole to jest oznaczone jako wymagane poprzez `Field(...)` i zawiera opis wyjaśniający jego przeznaczenie.

In [8]:
def parse_search_results(results_string: str) -> List[Tuple[str, str]]:
    try:
        return [
            (r.get("title", "Untitled"), r.get("link", ""))
            for r in json.loads(results_string)
        ]
    except json.JSONDecodeError:
        print("Error parsing search results. Returning empty list.")
        return []

In [10]:
def retrieve_documents(query: str, faiss_index: FAISS, k: int = 3) -> List[str]:
    docs = faiss_index.similarity_search(query, k=k)
    return [doc.page_content for doc in docs]

Ta funkcja `retrieve_documents` stanowi kluczowy element systemu wyszukiwania semantycznego, który wykorzystuje zaawansowaną technologię FAISS (Facebook AI Similarity Search) do znalezienia najbardziej odpowiednich fragmentów tekstu dla danego zapytania.

In [11]:
def evaluate_documents(query: str, documents: List[str], llm: BaseChatModel) -> List[float]:
    return [retrieval_evaluator(query, doc, llm) for doc in documents]

Ta funkcja `evaluate_documents` służy do oceny trafności dokumentów w odniesieniu do zapytania użytkownika. Jest to istotny etap w procesie wyszukiwania informacji, który pozwala na ustalenie, które dokumenty są najbardziej wartościowe w kontekście konkretnego zapytania.

In [13]:
def perform_web_search(query: str, llm: BaseChatModel) -> Tuple[List[str], List[Tuple[str, str]]]:
    rewritten_query = rewrite_query(query, llm)
    web_results = search.run(rewritten_query)
    web_knowledge = knowledge_refinement(web_results, llm)
    sources = parse_search_results(web_results)
    return web_knowledge, sources

Ta funkcja `perform_web_search` stanowi kompleksowy mechanizm przeszukiwania internetu, który integruje kilka wcześniej zdefiniowanych komponentów w celu znalezienia i przetworzenia informacji z sieci. Przyjrzyjmy się, jak działa ten proces krok po kroku.

In [14]:
def generate_response(
    query: str, knowledge: str, sources: List[Tuple[str, str]], llm: BaseChatModel
) -> str:
    response_prompt = PromptTemplate(
        input_variables=["query", "knowledge", "sources"],
        template="Based on the following knowledge, answer the query. Include the sources with their links (if available) at the end of your answer:\nQuery: {query}\nKnowledge: {knowledge}\nSources: {sources}\nAnswer:",
    )
    input_variables = {
        "query": query,
        "knowledge": knowledge,
        "sources": "\n".join(
            [f"{title}: {link}" if link else title for title, link in sources]
        ),
    }
    response_chain = response_prompt | llm
    return response_chain.invoke(input_variables).content


Ta funkcja `generate_response` jest odpowiedzialna za tworzenie ostatecznej odpowiedzi dla użytkownika, wykorzystując zgromadzoną wiedzę i źródła informacji. Jest to finałowy etap przetwarzania, który integruje wszystkie wcześniejsze kroki w spójną, informacyjną odpowiedź.

In [20]:
def crag_process(query: str, faiss_index: FAISS, llm: BaseChatModel) -> str:
    print(f"\nProcessing query: {query}")

    # Retrieve and evaluate documents
    retrieved_docs = retrieve_documents(query, faiss_index)
    eval_scores = evaluate_documents(query, retrieved_docs, llm)

    print(f"\nRetrieved {len(retrieved_docs)} documents")
    print(f"Evaluation scores: {eval_scores}")

    # Determine action based on evaluation scores
    max_score = max(eval_scores)
    sources = []

    if max_score > 0.7:
        print("\nAction: Correct - Using retrieved document")
        best_doc = retrieved_docs[eval_scores.index(max_score)]
        final_knowledge = best_doc
        sources.append(("Retrieved document", ""))
    elif max_score < 0.3:
        print("\nAction: Incorrect - Performing web search")
        final_knowledge, sources = perform_web_search(query, llm)
    else:
        print("\nAction: Ambiguous - Combining retrieved document and web search")
        best_doc = retrieved_docs[eval_scores.index(max_score)]
        # Refine the retrieved knowledge
        retrieved_knowledge = knowledge_refinement(best_doc, llm)
        web_knowledge, web_sources = perform_web_search(query, llm)
        final_knowledge = "\n".join(retrieved_knowledge + web_knowledge)
        sources = [("Retrieved document", "")] + web_sources

    print("\nFinal knowledge:")
    print(final_knowledge)

    print("\nSources:")
    for title, link in sources:
        print(f"{title}: {link}" if link else title)

    # Generate response
    print("\nGenerating response...")
    response = generate_response(query, final_knowledge, sources, llm)

    print("\nResponse generated")
    return response

Ta funkcja `crag_process` "Context Retrieval Augmented Generation" - podejścia łączącego wyszukiwanie kontekstu z generowaniem tekstu.

### Etap 1: Wyszukiwanie i ocena dokumentów
### Etap 2: Wybór strategii odpowiedzi
### Etap 3: Łączenie i prezentacja informacji
### Etap 4: Generowanie odpowiedzi

### Znaczenie i zastosowanie

Ta funkcja reprezentuje nowoczesne podejście do systemów pytanie-odpowiedź, które:

1. **Łączy lokalne i internetowe źródła informacji** - system nie jest ograniczony tylko do jednego dokumentu, ale może sięgać do internetu, gdy jest to potrzebne.

2. **Podejmuje inteligentne decyzje** - zamiast zawsze stosować to samo podejście, system analizuje trafność dokumentów i dostosowuje strategię.

3. **Zapewnia transparentność** - system zachowuje informacje o źródłach i dołącza je do odpowiedzi.

4. **Jest adaptowalny** - w zależności od zapytania, system może polegać głównie na dokumencie, głównie na internecie lub na kombinacji obu źródeł.

# Test

### Setup Ollama Embeddings

In [16]:
embeddings = OllamaEmbeddings(model=CFG.model_embeddings_ollama)

### Setup OpenAi Embeddings

In [ ]:
#embedding = OpenAIEmbeddings()

### Setup embeddings

In [17]:
vectorstore = encode_pdf(CFG.path, embeddings=embeddings)

### Setup Ollama LLM

In [18]:
llm = ChatOllama(model=CFG.model_ollama, temperature=CFG.temperature, num_predict=CFG.max_tokens)

### Setup OpenAI LLM

In [ ]:
# llm = ChatOpenAI(model= CFG.model_openai, max_tokens = CFG.max_tokens, temperature = CFG.temperature)

In [21]:
query = "What are the main causes of climate change?"
result = crag_process(query, vectorstore, llm)
print(f"Query: {query}")
print(f"Answer: {result}")


Processing query: What are the main causes of climate change?

Retrieved 3 documents
Evaluation scores: [0.9, 0.8, 0.7]

Action: Correct - Using retrieved document

Final knowledge:
Chapter 2: Causes of Climate Change 
Greenhouse Gases 
The primary cause of recent climate change is the increase in greenhouse gases in the 
atmosphere. Greenhouse gases, such as carbon dioxide (CO2), methane (CH4), and nitrous 
oxide (N2O), trap heat from the sun, creating a "greenhouse effect." This effect is essential 
for life on Earth, as it keeps the planet warm enough to support life. However, human 
activities have intensified this natural process, leading to a warmer climate. 
Fossil Fuels 
Burning fossil fuels for energy releases large amounts of CO2. This includes coal, oil, and 
natural gas used for electricity, heating, and transportation. The industrial revolution marked 
the beginning of a significant increase in fossil fuel consumption, which continues to rise 
today. 
Coal

Sources:
Retri

In [23]:
query = "How did Harry Potter beat Quirinus Quirrell?"
result = crag_process(query, vectorstore, llm)
print(f"Query: {query}")
print(f"Answer: {result}")


Processing query: how did harry beat Quirinus Quirrell?

Retrieved 3 documents
Evaluation scores: [0.0, 0.0, 0.0]

Action: Incorrect - Performing web search
Error parsing search results. Returning empty list.

Final knowledge:
['- Quirrell found Voldemort and became possessed by him.', "- In the end, Quirrell died at Harry Potter's hands.", '- Quirrell was defeated by Harry through physical touch, which burned him.', "- According to the book, Quirrell cannot physically touch Harry due to Lily Potter's protective magic."]

Sources:

Generating response...

Response generated
Query: how did harry beat Quirinus Quirrell?
Answer: Harry managed to defeat Quirinus Quirrell by using a method that involved physical contact. This occurred because of the protective magic provided by Harry's mother, Lily Potter. The specific incident took place during the first year at Hogwarts when Voldemort had taken possession of Quirrell. Due to the protective enchantment cast by Lily Potter on her son, Quir